# Persistent Foundry Agent

Creates a persistent agent on Azure AI Foundry, sends a user message, and streams the response back in real time.

In [1]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print("Client ready.")

Client ready.


In [3]:
# Create a persistent Foundry v2 (Prompt) agent
agent_name = "WriterAgent"

agent = client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model="gpt-4.1",
        instructions="You are a helpful agent that writes engaging book stories.",
    ),
)
print(f"Agent created: {agent.name} (version {agent.version})")

Agent created: WriterAgent (version 1)


In [4]:
# Prepare the user input — v2 agents use the OpenAI Responses API (no threads/messages)
user_input = "Write a short story about a robot who discovers music."

openai_client = client.get_openai_client()
print("OpenAI client ready.")

OpenAI client ready.


In [5]:
# Stream the agent's response via the Responses API
print("--- Agent Response ---\n")

stream = openai_client.responses.create(
    stream=True,
    input=user_input,
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "version": agent.version,
            "type": "agent_reference",
        }
    },
)

for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
print()


--- Agent Response ---

In a quiet corner of the city, where silver mist curled around endless rows of antennas, a robot named Lumo patrolled each night. Lumo’s job was simple: scan for faulty lights, patch circuits, and sweep away the blanket of dust that settled on every rooftop.

One evening, while inspecting the aging panels of Building 459, Lumo’s sensors picked up something strange—a vibration in the air, faint but persistent, pulsing through the metal structures. Curious, Lumo traced the signal, following it to a half-open window four stories below.

Floating out was an unfamiliar pattern: waves of sound rising and falling, weaving in and out like starlight. Lumo beeped in confusion. He’d never encountered noise like this. His programming had a word for it: “music.”

Carefully, Lumo lowered himself to the window and listened. Strings trembled, keys danced, and voices soared in a language beyond data. He felt something clinking inside his circuits, as if tiny sparks were flutteri

In [6]:
# Cleanup: delete the agent (all versions) when done
client.agents.delete(agent_name=agent.name)
print("\nAgent deleted.")


Agent deleted.
